# Assignment 5

In [ ]:
# load libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid, solve_ivp

In [ ]:
# constants
k = 8.988e9 #N*m^2/C^-2
e = 1.602e-19 #C
m_electron = 9.109e-31 #kg
m_proton = 1.673e-27 #kg

## 1 Mapping the Vector Field

Electric Field $\vec{E}$

$$
\vec{E}  =  \frac{1}{4 \pi \epsilon_0} \int \frac{dq}{r^2} \hat{r}  =  k \int \frac{ \lambda dz }{ y^2 + z^2 } \hat{r}
$$

$$
\vec{E}  =  E_{y} \hat{j} + E_{z} \hat{k}  =  (E\sin{\theta})\hat{j} + (E\cos{\theta})\hat{k}
$$

$$
\vec{E}  =  \left( k \lambda \int \frac{ dz }{ y^2 + z^2 }  \frac{ y }{ \sqrt{y^2 + z^2} } \right) \hat{j} + \left( k \lambda \int \frac{ dz }{ y^2 + z^2 }  \frac{ z }{ \sqrt{y^2 + z^2} } \right) \hat{k}
$$

$$
\vec{E}  =  \left( k \lambda \int \frac{ y dz }{ (y^2 + z^2)^{3/2} } \right) \hat{j} + \left( k \lambda \int \frac{ z dz }{ (y^2 + z^2)^{3/2} }  \right) \hat{k}
$$


In [ ]:
#charged rod parameters
lam = 1e-9 #C/m, linear charge distribution
L = 1 #m, length of rod

# create function to evaluate electric field integral
# coordinate of test point (y,z)
def charged_rod_efield(y, z, L=L, lam=lam):

    #create integrand of electric field
    #z_rod is z coordinate of a point on rod
    def integrand(z_rod):
        r = np.sqrt(y**2 + (z - z_rod)**2)
        integrand_Ey = y/r**3
        integrand_Ez = (z - z_rod)/r**3
        return integrand_Ey, integrand_Ez
    
    #divide rod into n segments
    #make integrand for each rod contribution point
    n_segments = 100
    z_points = np.linspace(-L/2, L/2, n_segments)
    E_values = integrand(z_points)

    #calculates integral for the rod segment contribution
    #normalizes electric field at test point
    integral = trapezoid(E_values, z_points)
    Ey, Ez = k * lam * integral
    return Ey, Ez


In [ ]:
# creates array for y and z range
y_range = np.linspace(-5, 5, 20)
z_range = np.linspace(-5, 5, 20)

# extend values to grid; create tuple for test point coordinates (Y, Z)
Y, Z = np.meshgrid(y_range, z_range)

# initialize arrays for E field at test points
Ey_map = np.zeros(Y.shape)
Ez_map = np.zeros(Z.shape)

# input y and z coordinate into electric field calculator
# store electric field component for each point in array
for i in range(len(z_range)):
    for j in range(len(y_range)):
        Ey_map[i, j], Ez_map[i, j] = np.array(charged_rod_efield(Y[i,j], Z[i,j]))

In [ ]:
# plot electric field at every point
magnitude = np.sqrt(Y**2 + Z**2)
E_mag = np.sqrt(Ey_map**2 + Ez_map**2)

Ey = Ey_map/E_mag
Ez = Ez_map/E_mag

q_plot = plt.quiver(Y, Z, Ey, Ez, magnitude, cmap='plasma')
plt.title('Electric Field of Charged Rod')
plt.xlabel('Horizontal Distance (m)')
plt.ylabel('Vertical Distance (m)')

## 2 Comparing Proton and Electron Dynamics
deriving acceleration of charge in electric field of charged rod

$$
\vec{F} = q\vec{E} = m\vec{a} \\
\vec{a} = \frac{q}{m} \vec{E} \\

\vdots

$$

$$
a_{y}  =  - \frac{q}{m} \frac{k \lambda}{y} \left( \frac{z - L/2}{y \sqrt{(z - L/2)^2 + y^2}} + \frac{z + L/2}{y \sqrt{(z + L/2)^2 + y^2}} \right) \\
a_{z}  =  \frac{q}{m} k \lambda \left( \frac{ 1 }{\sqrt{(z - L/2)^2 + y^2}} - \frac{ 1 }{\sqrt{(z + L/2)^2 + y^2}} \right)
$$

system of differential equations for charge dynamics
$$
\frac{dx_{y}}{dt}  =  v_{y}  \qquad  \frac{dx_{z}}{dt}  =  v_{z} \\
\frac{dv_{y}}{dt}  =  a_{y}  \qquad  \frac{dv_{z}}{dt}  =  a_{z}
$$

In [ ]:
# create differential equations for proton dynamics
def proton_dynamics(t, state):
    #current state of proton
    y, z, vy, vz = state

    #make differential for acceleration
    r1 = np.sqrt(y**2 + (z - L/2)**2) 
    r2 = np.sqrt(y**2 + (z + L/2)**2) 
    proton_constant = k*lam*e/m_proton
    ay = -proton_constant/y * ( (z - L/2)/r1 - (z + L/2)/r2 )
    az = proton_constant * (1/r1 - 1/r2)

    #return differential equations
    return [vy, vz, ay, az]

# establish intial conditions and time span
t0 = [0.1, -0.2, 0, 5e5]
tmax = 1e-5
t_span = (0,tmax)
t_eval = np.linspace(0,tmax,100)

# use solve_ivp to solve position throughout time
proton_sol = solve_ivp(proton_dynamics, t_span, t0, t_eval=t_eval)

# assign steps to variable
proton_tstep = proton_sol.t
proton_ystep = proton_sol.y[0]
proton_zstep = proton_sol.y[1]

In [ ]:
# create differential equations for electron dynamics
def electron_dynamics(t, state):
    #current state of electron
    y, z, vy, vz = state

    #make differential equation for acceleration
    r1 = np.sqrt(y**2 + (z - L/2)**2) 
    r2 = np.sqrt(y**2 + (z + L/2)**2) 
    electron_constant = -k*lam*e/m_electron
    ay = -electron_constant/y * ( (z - L/2)/r1 - (z + L/2)/r2 )
    az = electron_constant * (1/r1 - 1/r2)

    #return differential equations
    return [vy, vz, ay, az]

# establish intial conditions and time span

t0 = [0.1, -0.2, 0, 5e5]
tmax = 2e-6
t_span = (0,tmax)
t_eval = np.linspace(0,tmax,100)

# use solve_ivp to solve position thoughout time
electron_sol = solve_ivp(electron_dynamics, t_span, t0, t_eval=t_eval)

# assign steps to variable
electron_tstep = electron_sol.t
electron_ystep = electron_sol.y[0]
electron_zstep = electron_sol.y[1]

In [ ]:
# plot proton and electron path
plt.plot(proton_ystep, proton_zstep, color='red', label=f'proton {tmax} s')
plt.plot(electron_ystep, electron_zstep, color='blue', label=f'electron {tmax} s')

q_plot = plt.quiver(Y, Z, Ey, Ez, magnitude, cmap='plasma')
plt.title('Electric Field of Charged Rod')
plt.xlabel('Horizontal Distance (m)')
plt.ylabel('Vertical Distance (m)')
plt.legend()

#### Q2.2
The path of electron looks more curved than the proton's path, because the electron has less mass than the proton. Therefore it experiences greater acceleration, by Newton's second law, leading to greater curvature as its postion and direction rapidly changes.

## 3 Verification

In [ ]:
# create function to calculate electric field of point charge
def point_charge_efield(y, z, q=lam):
    r = np.sqrt(y**2 + z**2)
    Ey, Ez = k*q*y/r**3, k*q*z/r**3
    Ey, Ez = float(Ey), float(Ez)
    return Ey, Ez

# test point coordinates
y1_test = 5 #m
z1_test = 0 #m

# comparing electric field of point charge to charged rod
point_charge_test = point_charge_efield(y1_test, z1_test)
print(f'electric field y={y1_test}m away from point charge:')
print(f'Ey={point_charge_test[0]}, Ez={point_charge_test[1]}')

charged_rod_test = charged_rod_efield(y1_test, z1_test)
print(f'electric field y={y1_test}m away from charged rod:')
print(f'Ey={charged_rod_test[0]}, Ez={charged_rod_test[1]}')

In [ ]:
# change number of segments of charged rod function
def charged_rod_efield_seg(y, z, n_segments, L=L, lam=lam):
    def integrand(z_rod):
        r = np.sqrt(y**2 + (z - z_rod)**2)
        integrand_Ey = y/r**3
        integrand_Ez = (z - z_rod)/r**3
        return integrand_Ey, integrand_Ez
    
    z_points = np.linspace(-L/2, L/2, n_segments)
    E_values = integrand(z_points)

    integral = trapezoid(E_values, z_points)
    Ey, Ez = k * lam * integral
    return Ey, Ez

# test point coordinates
y2_test = 0.001 #m
z2_test = 0 #m

inf_charged_rod_efield = 2*k*lam/y2_test
print(f'electric field y={y2_test}m away from *infinite* charged rod:')
print(f'Ey={inf_charged_rod_efield}, Ez={0}')

n_seg = 1000
charged_rod_test = charged_rod_efield_seg(y2_test, z2_test, n_seg)
print(f'electric field y={y2_test}m away from charged rod ({n_seg} segments):')
print(f'Ey={charged_rod_test[0]}, Ez={charged_rod_test[1]}')

n_seg = 1010
charged_rod_test = charged_rod_efield_seg(y2_test, z2_test, n_seg)
print(f'electric field y={y2_test}m away from charged rod ({n_seg} segments):')
print(f'Ey={charged_rod_test[0]}, Ez={charged_rod_test[1]}')


## 4 Physical Discussion

#### Q4.1
Charges are distributed the same along the central axis, so every slice of electric field is the same around the rod.

#### Q4.2
The simulation requires about 1010-1011 segments for the error to drop below 1%.

## 5 The Safety Zone

In [ ]:
# create differential equations for proton dynamics
def proton_dynamics(t, state):
    #current state of proton
    y, z, vy, vz = state

    #make differential for acceleration
    r1 = np.sqrt(y**2 + (z - L/2)**2) 
    r2 = np.sqrt(y**2 + (z + L/2)**2) 
    proton_constant = k*lam*e/m_proton
    ay = -proton_constant/y * ( (z - L/2)/r1 - (z + L/2)/r2 )
    az = proton_constant * (1/r1 - 1/r2)

    #return differential equations
    return [vy, vz, ay, az]

# establish intial conditions and time span
t0 = [0.15, -1, 0, 2e6]
tmax = 1e-6
t_span = (0,tmax)
t_eval = np.linspace(0,tmax,100)

# use solve_ivp to solve position throughout time
proton_sol = solve_ivp(proton_dynamics, t_span, t0, t_eval=t_eval)

# assign steps to variable
proton_tstep = proton_sol.t
proton_ystep = proton_sol.y[0]
proton_zstep = proton_sol.y[1]


# plot proton and electron path
plt.plot(proton_ystep, proton_zstep, color='black', label=f'proton')

q_plot = plt.quiver(Y, Z, Ey, Ez, magnitude, cmap='plasma')
plt.title('Electric Field of Charged Rod')
plt.xlabel('Horizontal Distance (m)')
plt.ylabel('Vertical Distance (m)')
plt.legend()

print(proton_ystep[75])
print(proton_zstep[75])

#### Q 5.1
Minimum initial velocity required so that distance from rod is within 1% of 15 cm is about $v_{0} = 2 \cdot 10^{6} \text{m/s}$.

## 6 The Dipole Sandbox

In [ ]:
#charged rod parameters
lam = 1e-8 #C/m, linear charge distribution
L = 0.20 #m, length of rod

# create function to evaluate electric field integral
# coordinate of test point (y,z)
def parallel_rod_efield(y, z, L=L, lam=lam):

    #create integrand of electric field
    #z_rod is z coordinate of a point on rod
    def integrand(z_rod):
        r_neg = np.sqrt( (y + 0.1)**2 + (z - z_rod)**2 )
        r_pos = np.sqrt( (y - 0.1)**2 + (z - z_rod)**2 )
        integrand_Ey = (y - 0.1)/r_pos**3 - (y + 0.1)/r_neg**3
        integrand_Ez = (z - z_rod)*(1/r_pos**3 - 1/r_neg**3)
        return integrand_Ey, integrand_Ez
    
    #divide rod into n segments
    #make integrand for each rod contribution point
    n_segments = 100
    z_points = np.linspace(-L/2, L/2, n_segments)
    E_values = integrand(z_points)

    #calculates integral for the rod segment contribution
    #normalizes electric field at test point
    integral = trapezoid(E_values, z_points)
    Ey, Ez = k * lam * integral
    return Ey, Ez


In [ ]:
# creates array for y and z range
y_range = np.linspace(-0.5, 0.5, 20)
z_range = np.linspace(-0.5, 0.5, 20)

# extend values to grid; create tuple for test point coordinates (Y, Z)
Y, Z = np.meshgrid(y_range, z_range)

# initialize arrays for E field at test points
Ey_map = np.zeros(Y.shape)
Ez_map = np.zeros(Z.shape)

# input y and z coordinate into electric field calculator
# store electric field component for each point in array
for i in range(len(z_range)):
    for j in range(len(y_range)):
        Ey_map[i, j], Ez_map[i, j] = np.array(parallel_rod_efield(Y[i,j], Z[i,j]))

In [ ]:
# create differential equations for proton dynamics
def proton_dynamics(t, state):
    #current state of proton
    y, z, vy, vz = state

    #make differential for acceleration
    r1_neg = np.sqrt((y + 0.1)**2 + (z - L/2)**2) 
    r2_neg = np.sqrt((y + 0.1)**2 + (z + L/2)**2)
    r1_pos = np.sqrt((y - 0.1)**2 + (z - L/2)**2) 
    r2_pos = np.sqrt((y - 0.1)**2 + (z + L/2)**2)  
    proton_constant = k*lam*e/m_proton
    ay = -proton_constant/(y - 0.1) * ( (z - L/2)/r1_pos - (z + L/2)/r2_pos ) + proton_constant/(y + 0.1) * ( (z - L/2)/r1_neg - (z + L/2)/r2_neg ) 
    az = proton_constant * (1/r1_pos - 1/r2_pos) - proton_constant * (1/r1_neg - 1/r2_neg)

    #return differential equations
    return [vy, vz, ay, az]

# establish intial conditions and time span
t0 = [0, -0.3, 0.001, 0]
tmax = 1.8e-5
t_span = (0,tmax)
t_eval = np.linspace(0,tmax,100)

# use solve_ivp to solve position throughout time
proton_sol_nudge = solve_ivp(proton_dynamics, t_span, t0, t_eval=t_eval)

# assign steps to variable
proton_tstep_nudge = proton_sol_nudge.t
proton_ystep_nudge = proton_sol_nudge.y[0]
proton_zstep_nudge = proton_sol_nudge.y[1]

In [ ]:
# establish intial conditions and time span
t0 = [0, -0.3, 3e5, 5e5]
tmax = 1e-6
t_span = (0,tmax)
t_eval = np.linspace(0,tmax,100)

# use solve_ivp to solve position throughout time
proton_sol_sling = solve_ivp(proton_dynamics, t_span, t0, t_eval=t_eval)

# assign steps to variable
proton_tstep_sling = proton_sol_sling.t
proton_ystep_sling = proton_sol_sling.y[0]
proton_zstep_sling = proton_sol_sling.y[1]

In [ ]:
# plot charged rods
plt.vlines(-0.1, -0.1, 0.1, color='blue', label ='neg rod')
plt.vlines(0.1, -0.1, 0.1, color='red', label='pos rod')

# plot proton path
plt.plot(proton_ystep_nudge, proton_zstep_nudge, color='black', label=f'nudged proton')
plt.plot(proton_ystep_sling, proton_zstep_sling, color='black', linestyle='--', label=f'slingshot proton')


# plot electric field at every point
magnitude = np.sqrt(Y**2 + Z**2)
E_mag = np.sqrt(Ey_map**2 + Ez_map**2)

Ey = Ey_map/E_mag
Ez = Ez_map/E_mag

q_plot = plt.quiver(Y, Z, Ey, Ez, magnitude, cmap='plasma')
plt.title('Electric Field of Charged Rod')
plt.xlabel('Horizontal Distance (m)')
plt.ylabel('Vertical Distance (m)')
plt.legend()

#### Q 6.4
I can not find a specific velocity where the proton sling shots around the positive rod and into the negative rod.
